<a href="https://colab.research.google.com/github/Vronska-Anhelina/Metrics-Analysis-A-B-testing-/blob/main/Metrics_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade google-cloud-bigquery
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np


auth.authenticate_user()
client = bigquery.Client(project="data-analytics-mate")


In [ ]:
query="""
WITH session_info AS(
    SELECT
    s.date,
    s.ga_session_id,
    ab.test,
    ab.test_group,
    sp.country,
    sp.device,
    sp.continent,
    sp.channel
FROM `DA.ab_test` ab
JOIN `DA.session` s
on ab.ga_session_id=s.ga_session_id
JOIN `DA.session_params` sp
ON sp.ga_session_id=ab.ga_session_id),










session_with_orders AS(
  SELECT  session_info.date,
    session_info.test,
    session_info.test_group,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    COUNT(DISTINCT o.ga_session_id) AS session_with_orders
  FROM `DA.order` o
JOIN session_info
ON o.ga_session_id=session_info.ga_session_id
GROUP BY session_info.date,
    session_info.test,
    session_info.test_group,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel),








events AS(
SELECT session_info.date,
    session_info.test,
    session_info.test_group,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    ep.event_name,
    count(ep.ga_session_id) AS event_cnt
FROM `DA.event_params` ep
JOIN session_info
ON ep.ga_session_id=session_info.ga_session_id
GROUP BY session_info.date,
    session_info.test,
    session_info.test_group,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    ep.event_name),










session AS(
  SELECT session_info.date,
   session_info.test,
    session_info.test_group,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
  FROM session_info
GROUP BY session_info.test,
    session_info.test_group,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
session_info.date),






account AS (


  SELECT session_info.test,
    session_info.test_group,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
session_info.date,
COUNT(DISTINCT acs.ga_session_id) AS account_cnt
  FROM `DA.account_session` acs
JOIN session_info
ON acs.ga_session_id=session_info.ga_session_id
GROUP BY session_info.test,
    session_info.test_group,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
session_info.date
)


SELECT
  session_with_orders.date,
  session_with_orders.country,
  session_with_orders.device,
  session_with_orders.continent,
  session_with_orders.channel,
  session_with_orders.test,
  session_with_orders.test_group,
  'session with orders' AS event_name,
  session_with_orders.session_with_orders AS value
FROM session_with_orders
UNION ALL
SELECT
  events.date,
  events.country,
  events.device,
  events.continent,
  events.channel,
  events.test,
  events.test_group,
  events.event_name,
  events.event_cnt AS value
FROM events
UNION ALL
SELECT
  session.date,
  session.country,
  session.device,
  session.continent,
  session.channel,
  session.test,
  session.test_group,
  'session' AS event_name,
  session.session_cnt AS value
FROM session
UNION ALL
SELECT
  account.date,
  account.country,
  account.device,
  account.continent,
  account.channel,
  account.test,
  account.test_group,
  'new account' AS event_name,
  account.account_cnt AS value
FROM account;"""
query_job = client.query(query)
results = query_job.result()
df = results.to_dataframe()
print(df)


              date      country   device continent         channel  test  \
0       2020-12-08    Palestine  desktop      Asia          Direct     4   
1       2020-12-08    Palestine  desktop      Asia          Direct     3   
2       2020-11-06  Puerto Rico  desktop  Americas   Social Search     2   
3       2020-11-06  Puerto Rico  desktop  Americas   Social Search     1   
4       2020-12-08      Croatia  desktop    Europe          Direct     4   
...            ...          ...      ...       ...             ...   ...   
800991  2020-12-18      Vietnam  desktop      Asia     Paid Search     3   
800992  2020-12-28      Vietnam   tablet      Asia     Paid Search     4   
800993  2021-01-15      Vietnam  desktop      Asia   Social Search     4   
800994  2021-01-16      Vietnam  desktop      Asia     Paid Search     4   
800995  2021-01-18      Vietnam   mobile      Asia  Organic Search     4   

        test_group       event_name  value  
0                2      new account      1

In [ ]:
print(df["event_name"].unique())

['new account' 'scroll' 'user_engagement' 'session_start' 'page_view'
 'first_visit' 'view_item' 'view_promotion' 'select_promotion'
 'view_search_results' 'begin_checkout' 'add_shipping_info' 'add_to_cart'
 'select_item' 'session with orders' 'add_payment_info' 'click' 'session'
 'view_item_list']


In [ ]:
print(df.columns.tolist())

['date', 'country', 'device', 'continent', 'channel', 'test', 'test_group', 'event_name', 'value']


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

metrics = {
    "add_payment_info": "session",
    "add_shipping_info": "session",
    "begin_checkout": "session",
    "new account": "session",
    "add_to_cart": "session",
    "view_item": "session",
    "select_item": "session",
    "session with orders": "session",
    "page_view": "session",
    "first_visit": "session",
}

results = []
for test_number in df["test"].unique():
    test_df = df[df["test"] == test_number]
    for event, denominator_event in metrics.items():
        numerator = test_df[test_df["event_name"] == event]
        denominator = test_df[test_df["event_name"] == denominator_event]

        num_control = numerator[numerator["test_group"] == 1]["value"].sum()
        den_control = denominator[denominator["test_group"] == 1]["value"].sum()

        num_test = numerator[numerator["test_group"] == 2]["value"].sum()
        den_test = denominator[denominator["test_group"] == 2]["value"].sum()
        if den_control == 0 or den_test == 0:
            continue

        conversion_rate_control = num_control / den_control
        conversion_rate_test = num_test / den_test

        count_total = [num_control, num_test]
        nobs = [den_control, den_test]
        z_stat, p_value = proportions_ztest(count_total, nobs)

        results.append({
            "test_number": test_number,
            "metric": f"{event} / {denominator_event}",
            "numerator_event": event,
            "denominator_event": denominator_event,
            "numerator_control": num_control,
            "denominator_control": den_control,
            "conversion_rate_control": conversion_rate_control,
            "numerator_test": num_test,
            "denominator_test": den_test,
            "conversion_rate_test": conversion_rate_test,
            "metric_change": ((conversion_rate_test - conversion_rate_control) / conversion_rate_control) * 100,
            "z_stat": z_stat,
            "p_value": p_value,
            "significant": p_value < 0.05
        })

results = pd.DataFrame(results)
print(results)

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)


    test_number                         metric      numerator_event  \
0             4     add_payment_info / session     add_payment_info   
1             4    add_shipping_info / session    add_shipping_info   
2             4       begin_checkout / session       begin_checkout   
3             4          new account / session          new account   
4             4          add_to_cart / session          add_to_cart   
5             4            view_item / session            view_item   
6             4          select_item / session          select_item   
7             4  session with orders / session  session with orders   
8             4            page_view / session            page_view   
9             4          first_visit / session          first_visit   
10            3     add_payment_info / session     add_payment_info   
11            3    add_shipping_info / session    add_shipping_info   
12            3       begin_checkout / session       begin_checkout   
13    

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)


In [ ]:
results.to_csv("/content/ab_test_results.csv", index=False, sep=";", encoding="utf-8")

from google.colab import files
files.download("/content/ab_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
results.to_csv("/content/drive/My Drive/ab_test_results.csv", index=False, encoding="utf-8")

https://drive.google.com/file/d/1upXBDN8blnbWYZHpT1oh1D7nlDjFlRr3/view?usp=sharing

https://public.tableau.com/views/ABtest_17764694794480/ABtest?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link